In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
!pip install wbgapi pandas pyarrow -q
# import worldbank_capacity

In [ ]:
#!/usr/bin/env python3
"""
worldbank_capacity.py  —  Google Colab
======================================
Tier-3 CAPACITY layer for Paper 1 (research-risk misalignment). Pulls World Bank
country-year indicators so that "under-researched relative to risk" can be
separated from the trivial "poor countries publish less." Without these controls
the misalignment finding is not defensible; with them it is.

INDICATORS (country x year):
  GB.XPD.RSDV.GD.ZS  R&D expenditure (% of GDP)
  SP.POP.SCIE.RD.P6  researchers in R&D (per million people)
  NY.GDP.MKTP.CD     GDP (current US$)
  NY.GDP.PCAP.CD     GDP per capita (current US$)
  SP.POP.TOTL        population (for per-capita normalizations & rates)
  SE.XPD.TOTL.GD.ZS  govt education expenditure (% GDP) -- optional capacity proxy

ALSO attaches World Bank income group + region (HIC/UMIC/LMIC/LIC) so the same
table serves Paper 3's income classification and the SSA/Maghreb blocs.

OUTPUT
  1. wb_capacity_country_year.parquet   long panel: iso3, iso2, year, indicators
  2. wb_country_meta.parquet            iso3, iso2, income_group, wb_region
  3. merged onto your corpus by (study country, oa_year) -> per-paper capacity
     of the place studied (a Paper-1 covariate) + a country-year panel for models

APPROACH: World Bank API via `wbgapi` (official, robust). Country-level only —
no rasters, no per-record crawl. Fast.

SETUP
  !pip install wbgapi pandas pyarrow -q
"""

import sys
import pandas as pd
from pathlib import Path

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
IN_PARQUET   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/ror_enrich.parquet"   # your latest merged parquet
OUT_PANEL    = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/external_data/wb_capacity_country_year.parquet"
OUT_META     = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/external_data/wb_country_meta.parquet"
OUT_MERGED   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_capacity.parquet"
YEAR_MIN     = 1990
YEAR_MAX     = 2027

Path(OUT_PANEL).parent.mkdir(parents=True, exist_ok=True)
Path(OUT_META).parent.mkdir(parents=True, exist_ok=True)
Path(OUT_MERGED).parent.mkdir(parents=True, exist_ok=True)

INDICATORS = {
    "GB.XPD.RSDV.GD.ZS": "rd_pct_gdp",
    "SP.POP.SCIE.RD.P6": "researchers_per_mil",
    "NY.GDP.MKTP.CD":    "gdp_usd",
    "NY.GDP.PCAP.CD":    "gdp_pc_usd",
    "SP.POP.TOTL":       "population",
    "SE.XPD.TOTL.GD.ZS": "edu_pct_gdp",
}


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)


# =========================================================================== #
# 1. pull the country-year panel                                              #
# =========================================================================== #
def fetch_panel():
    import wbgapi as wb
    frames = []
    for code, name in INDICATORS.items():
        print(f"[wb] fetching {name} ({code})", file=sys.stderr)
        # economy='all', long format, all years in range
        df = wb.data.DataFrame(code, time=range(YEAR_MIN, YEAR_MAX + 1),
                               labels=False, skipBlanks=False)
        # wb returns economies as rows, years as columns 'YR1990'...
        df = df.reset_index().melt(id_vars="economy", var_name="year", value_name=name)
        df["year"] = df["year"].str.replace("YR", "", regex=False).astype(int)
        df = df.rename(columns={"economy": "iso3"})
        frames.append(df.set_index(["iso3", "year"])[name])
    panel = pd.concat(frames, axis=1).reset_index()
    # drop aggregate 'economies' (regions/income groups) — keep real countries via meta
    print(f"[wb] panel rows: {len(panel):,}", file=sys.stderr)
    return panel


# =========================================================================== #
# 2. country metadata: iso2 crosswalk + income group + region                 #
# =========================================================================== #
def fetch_meta():
    import wbgapi as wb
    rows = []
    for e in wb.economy.list():
        # keep actual countries (aggregates have region == 'NA'/'')
        iso3 = e.get("id", "")
        rows.append({
            "iso3": iso3,
            "country_name": e.get("value", ""),
            "iso2": e.get("iso2code", "") or "",
            "income_group": e.get("incomeLevel", "") or "",
            "wb_region": e.get("region", "") or "",
            "is_aggregate": e.get("aggregate", False),
        })
    meta = pd.DataFrame(rows)
    meta = meta[~meta["is_aggregate"]].drop(columns="is_aggregate")
    print(f"[wb] {len(meta):,} countries in metadata", file=sys.stderr)
    return meta


# =========================================================================== #
# 3. attach to the corpus by (study country, publication year)                #
# =========================================================================== #
def to_iso3(iso2, iso2_to_iso3):
    return iso2_to_iso3.get(S(iso2).upper(), "")

def attach(df, panel, meta):
    iso2_to_iso3 = dict(zip(meta["iso2"].str.upper(), meta["iso3"]))
    panel_idx = panel.set_index(["iso3", "year"])

    def row(r):
        # primary study country (first listed); Paper 1 uses the place studied
        sc = S(r.get("study_countries")).split(";")
        sc = [c for c in sc if c]
        year = pd.to_numeric(r.get("oa_year"), errors="coerce")
        if not sc or pd.isna(year):
            return pd.Series({v: None for v in INDICATORS.values()} |
                             {"study_iso3": "", "study_income_group": ""})
        iso3 = to_iso3(sc[0], iso2_to_iso3)
        yr = int(year)
        # use the paper's year; if that year is missing, fall back to nearest <= year
        vals = {}
        for name in INDICATORS.values():
            v = None
            if (iso3, yr) in panel_idx.index:
                v = panel_idx.loc[(iso3, yr), name]
            if pd.isna(v):
                # nearest prior year with data (indicators are sparse)
                sub = panel_idx.loc[iso3] if iso3 in panel_idx.index.get_level_values(0) else None
                if sub is not None and name in sub:
                    prior = sub[name].loc[:yr].dropna()
                    v = prior.iloc[-1] if len(prior) else None
            vals[name] = v
        inc = meta.loc[meta["iso3"] == iso3, "income_group"]
        vals["study_iso3"] = iso3
        vals["study_income_group"] = inc.iloc[0] if len(inc) else ""
        return pd.Series(vals)

    add = df.apply(row, axis=1)
    out = pd.concat([df, add], axis=1)
    out.to_parquet(OUT_MERGED, index=False)
    print(f"[write] {OUT_MERGED}", file=sys.stderr)

    cov = out["study_iso3"].astype(str).ne("").mean()
    rd_cov = out["rd_pct_gdp"].notna().mean()
    print(f"[coverage] papers matched to a study country: {cov:.1%}; "
          f"with R&D% value: {rd_cov:.1%}", file=sys.stderr)
    return out


# =========================================================================== #
if __name__ == "__main__":
    panel = fetch_panel()
    meta = fetch_meta()
    panel.to_parquet(OUT_PANEL, index=False)
    meta.to_parquet(OUT_META, index=False)
    print(f"[write] {OUT_PANEL}\n[write] {OUT_META}", file=sys.stderr)

    df = pd.read_parquet(IN_PARQUET)
    print(f"[in] {len(df):,} rows", file=sys.stderr)
    attach(df, panel, meta)

In [ ]:
meta

In [ ]:
meta.describe()

In [ ]:
panel

In [ ]:
panel.describe()